# Notebook 05b: Grey Wolf Optimization (GWO)

This notebook extends Notebook 03 by keeping the baseline preprocessing contract fixed (selected features, optimal scaler per model, and Stratified 5-fold CV) and tuning model hyperparameters with Grey Wolf Optimization (GWO) on training data only.

Generated outputs:
- `GWO_trials_full.csv`
- `GWO_convergence.csv`
- `GWO_best_params.csv`
- `GWO_optimal_fold_scores.csv`
- `GWO_optimization_time.csv`
- `GWO_test_results.csv` (optional section enabled below)

In [5]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    log_loss,
    cohen_kappa_score,
    matthews_corrcoef,
)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

RANDOM_SEED = 42
N_TRIALS_PER_MODEL = 50
GWO_POPULATION_SIZE = 10
if N_TRIALS_PER_MODEL % GWO_POPULATION_SIZE != 0:
    raise ValueError('N_TRIALS_PER_MODEL must be divisible by GWO_POPULATION_SIZE for strict budget parity.')
N_GWO_ITERATIONS = N_TRIALS_PER_MODEL // GWO_POPULATION_SIZE
INCLUDE_FINAL_TEST_EVAL = True

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = (NOTEBOOK_DIR / '../results/tables').resolve()
STATE_DIR = RESULTS_DIR / 'gwo_state'
STATE_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_SPACE_PATH = RESULTS_DIR / 'unified_search_space_clean.csv'

TRIALS_FULL_PATH = RESULTS_DIR / 'gwo/GWO_trials_full.csv'
CONVERGENCE_PATH = RESULTS_DIR / 'gwo/GWO_convergence.csv'
BEST_PARAMS_PATH = RESULTS_DIR / 'gwo/GWO_best_params.csv'
FOLD_SCORES_PATH = RESULTS_DIR / 'gwo/GWO_optimal_fold_scores.csv'
OPT_TIME_PATH = RESULTS_DIR / 'gwo/GWO_optimization_time.csv'
TEST_RESULTS_PATH = RESULTS_DIR / 'gwo/GWO_test_results.csv'

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'RANDOM_SEED = {RANDOM_SEED}')
print('CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)')
print(f'Results directory: {RESULTS_DIR}')
print(f'Search space file: {SEARCH_SPACE_PATH.name}')
print(f'GWO budget: evaluations/model={N_TRIALS_PER_MODEL}, population={GWO_POPULATION_SIZE}, outer_iterations={N_GWO_ITERATIONS}')

RANDOM_SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Results directory: C:\Users\omar8\Desktop\2026\Master & Research\smart-grid-stability-v01\results\tables
Search space file: unified_search_space_clean.csv
GWO budget: evaluations/model=50, population=10, outer_iterations=5


In [2]:
required_files = [
    RESULTS_DIR / 'X_train.npy',
    RESULTS_DIR / 'X_test.npy',
    RESULTS_DIR / 'y_train.npy',
    RESULTS_DIR / 'y_test.npy',
    RESULTS_DIR / 'selected_features.json',
    RESULTS_DIR / 'feature_names.csv',
    RESULTS_DIR / 'train_test_split_info.json',
    RESULTS_DIR / 'baseline_optimal_scalers.csv',
    RESULTS_DIR / 'baseline_cv_summary.csv',
    RESULTS_DIR / 'optimal_scaler_fold_scores.csv',
    RESULTS_DIR / 'baseline_test_results_clean.csv',
    RESULTS_DIR / 'baseline_inference_time.csv',
    SEARCH_SPACE_PATH,
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing))

X_train = np.load(RESULTS_DIR / 'X_train.npy')
X_test = np.load(RESULTS_DIR / 'X_test.npy')
y_train = np.load(RESULTS_DIR / 'y_train.npy')
y_test = np.load(RESULTS_DIR / 'y_test.npy')

with open(RESULTS_DIR / 'selected_features.json', 'r', encoding='utf-8') as f:
    selected_features = json.load(f)

with open(RESULTS_DIR / 'train_test_split_info.json', 'r', encoding='utf-8') as f:
    split_info = json.load(f)

feature_names_df = pd.read_csv(RESULTS_DIR / 'feature_names.csv')
if 'feature' not in feature_names_df.columns:
    raise ValueError('feature_names.csv must contain a feature column')

all_feature_names = feature_names_df['feature'].tolist()
feature_to_idx = {name: idx for idx, name in enumerate(all_feature_names)}
selected_idx = [feature_to_idx[f] for f in selected_features if f in feature_to_idx]

if len(selected_idx) != len(selected_features):
    missing_features = sorted(set(selected_features) - set(feature_to_idx.keys()))
    raise ValueError(f'Selected features not found in feature_names.csv: {missing_features}')

X_train_sel = X_train[:, selected_idx]
X_test_sel = X_test[:, selected_idx]

baseline_scalers_df = pd.read_csv(RESULTS_DIR / 'baseline_optimal_scalers.csv')
required_scaler_cols = {'Model', 'Optimal_Scaler'}
if not required_scaler_cols.issubset(baseline_scalers_df.columns):
    raise ValueError(f'baseline_optimal_scalers.csv missing columns: {required_scaler_cols}')

baseline_cv_summary_df = pd.read_csv(RESULTS_DIR / 'baseline_cv_summary.csv')
optimal_scaler_fold_scores_df = pd.read_csv(RESULTS_DIR / 'optimal_scaler_fold_scores.csv')
baseline_test_results_df = pd.read_csv(RESULTS_DIR / 'baseline_test_results_clean.csv')
baseline_inference_time_df = pd.read_csv(RESULTS_DIR / 'baseline_inference_time.csv')

search_df = pd.read_csv(SEARCH_SPACE_PATH)
required_search_cols = {'Model', 'Hyperparameter', 'Type', 'Lower_Bound', 'Upper_Bound', 'Choices', 'Log_Scale'}
if not required_search_cols.issubset(search_df.columns):
    raise ValueError(f'{SEARCH_SPACE_PATH.name} missing columns: {required_search_cols}')

print(f'X_train_sel shape: {X_train_sel.shape}')
print(f'X_test_sel shape: {X_test_sel.shape}')
print(f'Selected features: {len(selected_features)}')
print(f'Train/Test split info random_state: {split_info.get("random_state", "NA")}')
print(f'Baseline references loaded: cv={len(baseline_cv_summary_df)}, fold_scores={len(optimal_scaler_fold_scores_df)}, test={len(baseline_test_results_df)}, inference={len(baseline_inference_time_df)}')

X_train_sel shape: (48000, 13)
X_test_sel shape: (12000, 13)
Selected features: 13
Train/Test split info random_state: 42
Baseline references loaded: cv=14, fold_scores=14, test=14, inference=14


In [3]:
scalers = {
    'Raw': None,
    'Standard': StandardScaler(),
    'MinMax': MinMaxScaler(),
    'Robust': RobustScaler(),
    'Power': PowerTransformer(),
    'Quantile': QuantileTransformer(output_distribution='normal', random_state=RANDOM_SEED),
}

model_alias_to_code = {
    'LogisticRegression': 'LR',
    'LDA': 'LDA',
    'QDA': 'QDA',
    'NaiveBayes': 'NB',
    'KNN': 'KNN',
    'LinearSVC': 'LinearSVC',
    'SVC': 'SVM',
    'AdaBoost': 'AdaBoost',
    'RandomForest': 'RF',
    'GradientBoosting': 'GB',
    'XGBoost': 'XGB',
    'LightGBM': 'LGBM',
    'CatBoost': 'CB',
    'SGD': 'SGD',
}

code_to_model_alias = {v: k for k, v in model_alias_to_code.items()}


def build_model(model_code: str, params: Dict[str, Any]) -> Any:
    p = dict(params)

    if model_code == 'LR':
        penalty = p.get('penalty', 'l2')
        solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
        return LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, solver=solver, **p)

    if model_code == 'LDA':
        return LinearDiscriminantAnalysis(**p)

    if model_code == 'QDA':
        return QuadraticDiscriminantAnalysis(**p)

    if model_code == 'NB':
        return GaussianNB(**p)

    if model_code == 'KNN':
        return KNeighborsClassifier(**p)

    if model_code == 'LinearSVC':
        if p.get('loss') == 'hinge':
            p['dual'] = True
        else:
            p['dual'] = 'auto'
        return LinearSVC(random_state=RANDOM_SEED, **p)

    if model_code == 'SVM':
        return SVC(probability=True, random_state=RANDOM_SEED, **p)

    if model_code == 'AdaBoost':
        if p.get('algorithm') == 'SAMME.R':
            p['algorithm'] = 'SAMME'
        return AdaBoostClassifier(random_state=RANDOM_SEED, **p)

    if model_code == 'RF':
        if p.get('max_features') == 'None':
            p['max_features'] = None
        return RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, **p)

    if model_code == 'GB':
        return GradientBoostingClassifier(random_state=RANDOM_SEED, **p)

    if model_code == 'XGB':
        return xgb.XGBClassifier(eval_metric='logloss', random_state=RANDOM_SEED, verbosity=0, n_jobs=-1, **p)

    if model_code == 'LGBM':
        return lgb.LGBMClassifier(random_state=RANDOM_SEED, verbose=-1, n_jobs=-1, **p)

    if model_code == 'CB':
        return CatBoostClassifier(verbose=0, random_state=RANDOM_SEED, **p)

    if model_code == 'SGD':
        return SGDClassifier(max_iter=1000, random_state=RANDOM_SEED, **p)

    raise ValueError(f'Unsupported model code: {model_code}')


def parse_model_search_space(search_space_df: pd.DataFrame) -> Dict[str, List[Dict[str, Any]]]:
    grouped: Dict[str, List[Dict[str, Any]]] = {}
    for alias_name, g in search_space_df.groupby('Model'):
        model_code = model_alias_to_code.get(alias_name)
        if model_code is None:
            continue

        entries: List[Dict[str, Any]] = []
        for _, row in g.iterrows():
            hp_type = str(row['Type']).strip().lower()
            log_scale_raw = row.get('Log_Scale', False)
            if isinstance(log_scale_raw, str):
                log_scale_flag = log_scale_raw.strip().lower() == 'true'
            else:
                log_scale_flag = bool(log_scale_raw)

            entry = {
                'name': str(row['Hyperparameter']).strip(),
                'type': hp_type,
                'low': row.get('Lower_Bound', np.nan),
                'high': row.get('Upper_Bound', np.nan),
                'choices': row.get('Choices', np.nan),
                'log_scale': log_scale_flag,
            }
            entries.append(entry)
        grouped[model_code] = entries
    return grouped


model_spaces = parse_model_search_space(search_df)
optimal_scaler_by_model = dict(zip(baseline_scalers_df['Model'], baseline_scalers_df['Optimal_Scaler']))

target_models = [m for m in baseline_scalers_df['Model'].tolist() if m in model_spaces]
missing_spaces = [m for m in baseline_scalers_df['Model'].tolist() if m not in model_spaces]
if missing_spaces:
    print('Models with no search-space rows (will be skipped):', missing_spaces)


def make_pipeline(model_code: str, model_params: Dict[str, Any], scaler_name: str) -> Pipeline:
    model = build_model(model_code=model_code, params=model_params)
    scaler_obj = scalers[scaler_name]

    if scaler_obj is None:
        return Pipeline([('model', model)])

    return Pipeline([('scaler', clone(scaler_obj)), ('model', model)])


def _clip01(x: float) -> float:
    return float(min(1.0, max(0.0, x)))


def decode_position(position: np.ndarray, model_code: str) -> Dict[str, Any]:
    params: Dict[str, Any] = {}
    for i, entry in enumerate(model_spaces.get(model_code, [])):
        name = entry['name']
        typ = entry['type']
        raw = _clip01(float(position[i]))

        if typ == 'categorical':
            choices_raw = str(entry['choices'])
            choices = [c.strip() for c in choices_raw.split('|') if c.strip()]
            if not choices:
                raise ValueError(f'No categorical choices for {model_code}:{name}')
            idx = int(round(raw * (len(choices) - 1)))
            idx = min(max(idx, 0), len(choices) - 1)
            params[name] = choices[idx]
        elif typ == 'int':
            low = int(float(entry['low']))
            high = int(float(entry['high']))
            val = int(round(low + raw * (high - low)))
            params[name] = min(max(val, low), high)
        elif typ == 'float':
            low = float(entry['low'])
            high = float(entry['high'])
            if entry['log_scale']:
                if low <= 0 or high <= 0:
                    raise ValueError(f'Log-scale bounds must be > 0 for {model_code}:{name}')
                val = np.exp(np.log(low) + raw * (np.log(high) - np.log(low)))
            else:
                val = low + raw * (high - low)
            params[name] = float(val)
        else:
            raise ValueError(f'Unsupported hyperparameter type: {typ} for {model_code}:{name}')

    return params


print(f'Models to optimize: {target_models}')
print(f'Number of evaluations/model: {N_TRIALS_PER_MODEL}')

Models to optimize: ['CB', 'XGB', 'SVM', 'LGBM', 'RF', 'KNN', 'GB', 'LinearSVC', 'LDA', 'LR', 'QDA', 'SGD', 'AdaBoost', 'NB']
Number of evaluations/model: 50


In [4]:
def load_or_init_trials_df(path: Path) -> pd.DataFrame:
    cols = ['Model', 'Iteration', 'CV_Mean', 'Best_Score_So_Far', 'Iteration_Runtime_Sec']
    if path.exists():
        df = pd.read_csv(path)
        for c in cols:
            if c not in df.columns:
                raise ValueError(f'{path.name} missing required column: {c}')
        return df[cols].copy()
    return pd.DataFrame(columns=cols)


def save_trials_and_convergence(trials_df: pd.DataFrame) -> None:
    if trials_df.empty:
        trials_df.to_csv(TRIALS_FULL_PATH, index=False)
        pd.DataFrame(columns=['Model', 'Iteration', 'Best_Score_So_Far']).to_csv(CONVERGENCE_PATH, index=False)
        return

    clean = trials_df.drop_duplicates(subset=['Model', 'Iteration'], keep='last').sort_values(['Model', 'Iteration'])
    clean.to_csv(TRIALS_FULL_PATH, index=False)

    conv = clean[['Model', 'Iteration', 'Best_Score_So_Far']].copy()
    conv.to_csv(CONVERGENCE_PATH, index=False)


def load_or_init_best_params(path: Path) -> pd.DataFrame:
    fixed_cols = ['Model', 'Optimal_Scaler', 'Best_CV_Mean', 'Best_Iteration']
    if path.exists():
        df = pd.read_csv(path)
        for c in fixed_cols:
            if c not in df.columns:
                raise ValueError(f'{path.name} missing required column: {c}')
        return df
    return pd.DataFrame(columns=fixed_cols)


def upsert_best_row(best_df: pd.DataFrame, row: Dict[str, Any]) -> pd.DataFrame:
    row_df = pd.DataFrame([row])
    if best_df.empty:
        return row_df

    all_cols = sorted(set(best_df.columns) | set(row_df.columns))
    best_df = best_df.reindex(columns=all_cols)
    row_df = row_df.reindex(columns=all_cols)

    best_df = best_df[best_df['Model'] != row['Model']]
    out = pd.concat([best_df, row_df], ignore_index=True)
    out = out.sort_values('Model').reset_index(drop=True)
    return out


def state_path(model_code: str) -> Path:
    return STATE_DIR / f'GWO_state_{model_code}.json'


def _score_position(model_code: str, scaler_name: str, position: np.ndarray) -> tuple[float, Dict[str, Any], float]:
    t0 = time.perf_counter()
    params = decode_position(position, model_code=model_code)
    pipeline = make_pipeline(model_code=model_code, model_params=params, scaler_name=scaler_name)

    cv_out = cross_validate(
        pipeline,
        X_train_sel,
        y_train,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        return_train_score=False,
    )

    mean_score = float(np.mean(cv_out['test_score']))
    runtime_sec = float(time.perf_counter() - t0)
    return mean_score, params, runtime_sec


def _save_state(path: Path, state: Dict[str, Any]) -> None:
    serializable = {
        'wolves': state['wolves'].tolist(),
        'alpha_pos': state['alpha_pos'].tolist(),
        'beta_pos': state['beta_pos'].tolist(),
        'delta_pos': state['delta_pos'].tolist(),
        'alpha_score': float(state['alpha_score']),
        'beta_score': float(state['beta_score']),
        'delta_score': float(state['delta_score']),
        'alpha_params': state['alpha_params'],
        'best_iteration': int(state['best_iteration']),
        'outer_iter': int(state['outer_iter']),
        'wolf_idx': int(state['wolf_idx']),
        'eval_counter': int(state['eval_counter']),
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)


def _load_state(path: Path) -> Dict[str, Any]:
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    return {
        'wolves': np.array(raw['wolves'], dtype=float),
        'alpha_pos': np.array(raw['alpha_pos'], dtype=float),
        'beta_pos': np.array(raw['beta_pos'], dtype=float),
        'delta_pos': np.array(raw['delta_pos'], dtype=float),
        'alpha_score': float(raw['alpha_score']),
        'beta_score': float(raw['beta_score']),
        'delta_score': float(raw['delta_score']),
        'alpha_params': dict(raw.get('alpha_params', {})),
        'best_iteration': int(raw.get('best_iteration', 0)),
        'outer_iter': int(raw['outer_iter']),
        'wolf_idx': int(raw['wolf_idx']),
        'eval_counter': int(raw['eval_counter']),
    }


def _init_state(model_code: str) -> Dict[str, Any]:
    dim = len(model_spaces.get(model_code, []))
    if dim == 0:
        raise ValueError(f'No search-space dimensions for {model_code}')

    rng = np.random.default_rng(RANDOM_SEED)
    wolves = rng.random((GWO_POPULATION_SIZE, dim))
    return {
        'wolves': wolves,
        'alpha_pos': wolves[0].copy(),
        'beta_pos': wolves[1].copy() if GWO_POPULATION_SIZE > 1 else wolves[0].copy(),
        'delta_pos': wolves[2].copy() if GWO_POPULATION_SIZE > 2 else wolves[0].copy(),
        'alpha_score': float('-inf'),
        'beta_score': float('-inf'),
        'delta_score': float('-inf'),
        'alpha_params': {},
        'best_iteration': 0,
        'outer_iter': 0,
        'wolf_idx': 0,
        'eval_counter': 0,
    }


trials_df = load_or_init_trials_df(TRIALS_FULL_PATH)
best_params_df = load_or_init_best_params(BEST_PARAMS_PATH)

for model_code in target_models:
    scaler_name = optimal_scaler_by_model[model_code]

    model_trials = trials_df[trials_df['Model'] == model_code].sort_values('Iteration')
    completed = int(model_trials['Iteration'].nunique())

    if completed >= N_TRIALS_PER_MODEL and (best_params_df['Model'] == model_code).any():
        print(f'Skip {model_code}: already complete ({completed}/{N_TRIALS_PER_MODEL})')
        continue

    print(f'Optimizing {model_code} with scaler={scaler_name} | completed={completed}/{N_TRIALS_PER_MODEL}')

    s_path = state_path(model_code)
    if s_path.exists():
        state = _load_state(s_path)
    else:
        state = _init_state(model_code)

    if completed > 0 and not s_path.exists():
        raise RuntimeError(f'Missing state file for interrupted model {model_code}: {s_path.name}')

    if state['eval_counter'] != completed:
        raise RuntimeError(
            f'State mismatch for {model_code}: state eval_counter={state["eval_counter"]}, csv completed={completed}'
        )

    while state['eval_counter'] < N_TRIALS_PER_MODEL:
        wolf_idx = state['wolf_idx']
        outer_iter = state['outer_iter']
        a = 2.0 - 2.0 * (outer_iter / max(N_GWO_ITERATIONS - 1, 1))

        pos = state['wolves'][wolf_idx].copy()
        score, params, runtime_sec = _score_position(model_code=model_code, scaler_name=scaler_name, position=pos)

        if score > state['alpha_score']:
            state['delta_score'], state['delta_pos'] = state['beta_score'], state['beta_pos'].copy()
            state['beta_score'], state['beta_pos'] = state['alpha_score'], state['alpha_pos'].copy()
            state['alpha_score'], state['alpha_pos'] = score, pos.copy()
            state['alpha_params'] = dict(params)
            state['best_iteration'] = state['eval_counter'] + 1
        elif score > state['beta_score']:
            state['delta_score'], state['delta_pos'] = state['beta_score'], state['beta_pos'].copy()
            state['beta_score'], state['beta_pos'] = score, pos.copy()
        elif score > state['delta_score']:
            state['delta_score'], state['delta_pos'] = score, pos.copy()

        new_row = {
            'Model': model_code,
            'Iteration': state['eval_counter'] + 1,
            'CV_Mean': score,
            'Best_Score_So_Far': state['alpha_score'],
            'Iteration_Runtime_Sec': runtime_sec,
        }
        trials_df.loc[len(trials_df)] = new_row
        save_trials_and_convergence(trials_df)

        updated_pos = pos.copy()
        for d in range(updated_pos.shape[0]):
            r1, r2, r3 = np.random.rand(), np.random.rand(), np.random.rand()
            r4, r5, r6 = np.random.rand(), np.random.rand(), np.random.rand()

            A1 = 2.0 * a * r1 - a
            C1 = 2.0 * r2
            A2 = 2.0 * a * r3 - a
            C2 = 2.0 * r4
            A3 = 2.0 * a * r5 - a
            C3 = 2.0 * r6

            D_alpha = abs(C1 * state['alpha_pos'][d] - updated_pos[d])
            D_beta = abs(C2 * state['beta_pos'][d] - updated_pos[d])
            D_delta = abs(C3 * state['delta_pos'][d] - updated_pos[d])

            X1 = state['alpha_pos'][d] - A1 * D_alpha
            X2 = state['beta_pos'][d] - A2 * D_beta
            X3 = state['delta_pos'][d] - A3 * D_delta

            updated_pos[d] = _clip01((X1 + X2 + X3) / 3.0)

        state['wolves'][wolf_idx] = updated_pos

        state['eval_counter'] += 1
        state['wolf_idx'] += 1
        if state['wolf_idx'] >= GWO_POPULATION_SIZE:
            state['wolf_idx'] = 0
            state['outer_iter'] += 1

        _save_state(s_path, state)

    best_row: Dict[str, Any] = {
        'Model': model_code,
        'Optimal_Scaler': scaler_name,
        'Best_CV_Mean': float(state['alpha_score']),
        'Best_Iteration': int(state['best_iteration']),
    }
    best_row.update(state['alpha_params'])

    best_params_df = upsert_best_row(best_params_df, best_row)
    best_params_df.to_csv(BEST_PARAMS_PATH, index=False)

    print(f'  done {model_code} | best={state["alpha_score"]:.6f} at iter={state["best_iteration"]}')

save_trials_and_convergence(trials_df)
best_params_df.to_csv(BEST_PARAMS_PATH, index=False)

print('Saved:')
print(f' - {TRIALS_FULL_PATH.name}')
print(f' - {CONVERGENCE_PATH.name}')
print(f' - {BEST_PARAMS_PATH.name}')

Optimizing CB with scaler=Raw | completed=0/50
  done CB | best=0.995479 at iter=30
Optimizing XGB with scaler=Power | completed=0/50
  done XGB | best=0.990500 at iter=50
Optimizing SVM with scaler=Standard | completed=0/50
  done SVM | best=0.994458 at iter=33
Optimizing LGBM with scaler=Raw | completed=0/50
  done LGBM | best=0.997521 at iter=45
Optimizing RF with scaler=MinMax | completed=0/50
  done RF | best=0.973250 at iter=36
Optimizing KNN with scaler=Raw | completed=0/50
  done KNN | best=0.948792 at iter=28
Optimizing GB with scaler=MinMax | completed=0/50
  done GB | best=0.996542 at iter=42
Optimizing LinearSVC with scaler=Power | completed=0/50
  done LinearSVC | best=0.897062 at iter=41
Optimizing LDA with scaler=Power | completed=0/50
  done LDA | best=0.895312 at iter=1
Optimizing LR with scaler=Power | completed=0/50
  done LR | best=0.897375 at iter=20
Optimizing QDA with scaler=Power | completed=0/50
  done QDA | best=0.899125 at iter=38
Optimizing SGD with scaler=P

In [6]:
best_params_df = pd.read_csv(BEST_PARAMS_PATH)

base_cols = {'Model', 'Optimal_Scaler', 'Best_CV_Mean', 'Best_Iteration'}
float_keys = {
    'learning_rate', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda',
    'l2_leaf_reg', 'alpha', 'l1_ratio', 'reg_param', 'var_smoothing', 'C', 'coef0'
}

fold_rows: List[Dict[str, Any]] = []
for _, row in best_params_df.iterrows():
    model_code = row['Model']
    scaler_name = row['Optimal_Scaler']

    params: Dict[str, Any] = {}
    for col in best_params_df.columns:
        if col in base_cols:
            continue
        value = row[col]
        if pd.notna(value):
            params[col] = value

    for k, v in list(params.items()):
        if isinstance(v, float) and v.is_integer() and k not in float_keys:
            params[k] = int(v)

    pipeline = make_pipeline(model_code=model_code, model_params=params, scaler_name=scaler_name)
    cv_out = cross_validate(
        pipeline,
        X_train_sel,
        y_train,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        return_train_score=False,
    )

    fold_scores = cv_out['test_score']
    fold_row: Dict[str, Any] = {
        'Model': model_code,
        'Optimal_Scaler': scaler_name,
    }
    for i, s in enumerate(fold_scores, start=1):
        fold_row[f'Fold_{i}'] = float(s)

    fold_row['Mean'] = float(np.mean(fold_scores))
    fold_row['Std'] = float(np.std(fold_scores, ddof=0))
    fold_rows.append(fold_row)

fold_scores_df = pd.DataFrame(fold_rows).sort_values('Model').reset_index(drop=True)
fold_scores_df.to_csv(FOLD_SCORES_PATH, index=False)

trials_df = pd.read_csv(TRIALS_FULL_PATH)
opt_time_df = (
    trials_df.groupby('Model', as_index=False)
    .agg(
        Number_of_Iterations=('Iteration', 'count'),
        Total_Optimization_Time_Sec=('Iteration_Runtime_Sec', 'sum'),
        Mean_Iteration_Time_Sec=('Iteration_Runtime_Sec', 'mean'),
    )
    .sort_values('Model')
    .reset_index(drop=True)
)
opt_time_df.to_csv(OPT_TIME_PATH, index=False)

print(f'Saved: {FOLD_SCORES_PATH.name} ({len(fold_scores_df)} rows)')
print(f'Saved: {OPT_TIME_PATH.name} ({len(opt_time_df)} rows)')

Saved: GWO_optimal_fold_scores.csv (14 rows)
Saved: GWO_optimization_time.csv (14 rows)


In [7]:
if INCLUDE_FINAL_TEST_EVAL:
    best_params_df = pd.read_csv(BEST_PARAMS_PATH)
    base_cols = {'Model', 'Optimal_Scaler', 'Best_CV_Mean', 'Best_Iteration'}

    test_rows: List[Dict[str, Any]] = []
    for _, row in best_params_df.iterrows():
        model_code = row['Model']
        scaler_name = row['Optimal_Scaler']

        params: Dict[str, Any] = {}
        for col in best_params_df.columns:
            if col in base_cols:
                continue
            value = row[col]
            if pd.notna(value):
                params[col] = value

        for k, v in list(params.items()):
            if isinstance(v, float) and v.is_integer() and k not in float_keys:
                params[k] = int(v)

        model = build_model(model_code=model_code, params=params)
        scaler_obj = scalers[scaler_name]

        if scaler_obj is None:
            Xtr = X_train_sel
            Xte = X_test_sel
        else:
            s = clone(scaler_obj)
            Xtr = s.fit_transform(X_train_sel)
            Xte = s.transform(X_test_sel)

        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)

        acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
        kappa = cohen_kappa_score(y_test, y_pred)
        mcc = matthews_corrcoef(y_test, y_pred)

        if hasattr(model, 'predict_proba'):
            y_score = model.predict_proba(Xte)[:, 1]
            auc = roc_auc_score(y_test, y_score)
            ll = log_loss(y_test, y_score)
        elif hasattr(model, 'decision_function'):
            y_score = model.decision_function(Xte)
            auc = roc_auc_score(y_test, y_score)
            ll = np.nan
        else:
            auc = np.nan
            ll = np.nan

        test_rows.append({
            'Model': model_code,
            'Optimal_Scaler': scaler_name,
            'Accuracy': float(acc),
            'Precision': float(prec),
            'Recall': float(rec),
            'F1': float(f1),
            'AUC': float(auc) if pd.notna(auc) else np.nan,
            'LogLoss': float(ll) if pd.notna(ll) else np.nan,
            'Kappa': float(kappa),
            'MCC': float(mcc),
        })

    gwo_test_df = pd.DataFrame(test_rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)
    gwo_test_df.to_csv(TEST_RESULTS_PATH, index=False)
    print(f'Saved: {TEST_RESULTS_PATH.name} ({len(gwo_test_df)} rows)')
    print('Metrics columns: Accuracy, Precision, Recall, F1, AUC, LogLoss, Kappa, MCC')
else:
    print('Final test evaluation skipped by configuration.')

c:\Users\omar8\miniconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
c:\Users\omar8\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\omar8\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved: GWO_test_results.csv (14 rows)
Metrics columns: Accuracy, Precision, Recall, F1, AUC, LogLoss, Kappa, MCC


In [8]:
expected_outputs = [
    TRIALS_FULL_PATH,
    CONVERGENCE_PATH,
    BEST_PARAMS_PATH,
    FOLD_SCORES_PATH,
    OPT_TIME_PATH,
]
if INCLUDE_FINAL_TEST_EVAL:
    expected_outputs.append(TEST_RESULTS_PATH)

print('Generated outputs:')
for p in expected_outputs:
    status = 'OK' if p.exists() else 'MISSING'
    rows = len(pd.read_csv(p)) if p.exists() else 0
    print(f' - {p.name}: {status}, rows={rows}')

Generated outputs:
 - GWO_trials_full.csv: OK, rows=700
 - GWO_convergence.csv: OK, rows=700
 - GWO_best_params.csv: OK, rows=14
 - GWO_optimal_fold_scores.csv: OK, rows=14
 - GWO_optimization_time.csv: OK, rows=14
 - GWO_test_results.csv: OK, rows=14
